# 07 — Bison Habitat Suitability Index

**Project:** Pine Ridge Bison Habitat Suitability Analysis  
**Partner:** Oglala Lakota College (OLC) / Cubedynamics  
**Author:** Lilly Jones, PhD — Daear Consulting, LLC / CIRES Earth Lab

---

## What This Notebook Does

This is the synthesis notebook. It combines the five component layers
built in notebooks 02–06 into the Bison Habitat Suitability Index (BHSI),
then uses DBSCAN clustering to identify and rank priority restoration units.

**Two outputs:**

1. **BHSI raster** (GeoTIFF) — continuous 0–1 score across the full
   Pine Ridge Reservation. Higher score = better current or restorable
   bison habitat.

2. **Priority restoration units** (GeoPackage + CSV) — contiguous
   high-BHSI patches identified by DBSCAN clustering, ranked by
   composite quality score, with area, water access, soils, and
   land cover summaries.

## Algorithm: DBSCAN

See `documents/methods_clustering.md` for the full rationale.
Key parameters (all in `src/constants.py`):
- `DBSCAN_EPS_M = 1000` — max distance between pixels in same patch (meters)
- `DBSCAN_MIN_ACRES = 500` — minimum viable patch size
- `BHSI_THRESHOLD_PCT = 70` — top N% of scores classified as high-suitability

Review `DBSCAN_MIN_ACRES` with the OLC bison program before finalizing.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
from sklearn.cluster import DBSCAN

from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED, TARGET_RES_M,
    BHSI_WEIGHTS, DBSCAN_EPS_M, DBSCAN_MIN_ACRES, BHSI_THRESHOLD_PCT,
    CACHE_DIR, OUTPUTS_DIR, FIGURES_DIR,
)
from src.raster_utils import combine_bhsi_layers
from src.validation import validate_bhsi_layers, retain_minimum_area_clusters
from src.provenance import write_bhsi_manifest
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore")
%matplotlib inline

TEMPLATE_PATH = CACHE_DIR / "template_30m_albers.tif"
assert TEMPLATE_PATH.exists(), "Run notebook 01 first."
pine_ridge = gpd.read_file(OUTPUTS_DIR / "pine_ridge_boundary.geojson")

# Check which component layers exist
LAYER_PATHS = {
    "vegetation":  OUTPUTS_DIR / "bhsi_vegetation.tif",
    "soils":       OUTPUTS_DIR / "bhsi_soils.tif",
    "topography":  OUTPUTS_DIR / "bhsi_topography.tif",
    "water":       OUTPUTS_DIR / "bhsi_water.tif",
    "climate":     OUTPUTS_DIR / "bhsi_climate.tif",
}

available = {k: v for k, v in LAYER_PATHS.items() if v.exists()}
missing   = [k for k in LAYER_PATHS if k not in available]

print(f"Available layers: {list(available.keys())}")
if missing:
    raise FileNotFoundError(f"Missing required layers: {missing}. Run notebooks 02–06 first.")
if not np.isclose(sum(BHSI_WEIGHTS.values()), 1.0):
    raise ValueError("BHSI weights must sum to 1.0.")
validate_bhsi_layers(available, set(BHSI_WEIGHTS))
print(f"\nBHSI weights: {BHSI_WEIGHTS}")

In [ ]:
# Print the data sovereignty statement at the top of every notebook
print_data_acknowledgment()

---
## 1. Combine Component Layers → BHSI

In [ ]:
bhsi_path = OUTPUTS_DIR / "bhsi_composite.tif"

bhsi_array = combine_bhsi_layers(
    layer_paths  = available,
    weights      = BHSI_WEIGHTS,
    output_path  = bhsi_path,
)

valid = bhsi_array[~np.isnan(bhsi_array)]
print(f"BHSI composite statistics:")
print(f"  Valid pixels  : {len(valid):,}")
print(f"  Min           : {valid.min():.3f}")
print(f"  Max           : {valid.max():.3f}")
print(f"  Mean          : {valid.mean():.3f}")
print(f"  Median        : {np.median(valid):.3f}")
print(f"  75th pct      : {np.percentile(valid, 75):.3f}")
print(f"  90th pct      : {np.percentile(valid, 90):.3f}")
print(f"\nSaved: outputs/bhsi_composite.tif")

---
## 2. DBSCAN Clustering — Priority Restoration Units

In [ ]:
# Get coordinates of high-suitability pixels
threshold  = np.nanpercentile(bhsi_array, BHSI_THRESHOLD_PCT)
high_suit  = (bhsi_array > threshold) & (~np.isnan(bhsi_array))
rows, cols = np.where(high_suit)

print(f"BHSI threshold (top {100-BHSI_THRESHOLD_PCT}%): {threshold:.3f}")
print(f"High-suitability pixels: {len(rows):,} "
      f"({len(rows)/len(valid)*100:.1f}% of reservation)")

# Convert pixel indices to projected coordinates (meters)
with rasterio.open(TEMPLATE_PATH) as tmpl:
    transform = tmpl.transform

xs = transform.c + cols * transform.a
ys = transform.f + rows * transform.e
coords = np.column_stack([xs, ys])

# Compute min_samples from acres threshold
pixel_area_acres = (TARGET_RES_M ** 2) / 4046.86
min_samples      = max(10, int(DBSCAN_MIN_ACRES / pixel_area_acres))

print(f"\nDBSCAN parameters:")
print(f"  eps         : {DBSCAN_EPS_M:,}m ({DBSCAN_EPS_M/1000:.1f}km)")
print(f"  min_samples : {min_samples:,} pixels = ~{DBSCAN_MIN_ACRES:.0f} acres")
print(f"\nRunning DBSCAN (may take 30-60 seconds for large datasets)...")

db     = DBSCAN(eps=DBSCAN_EPS_M, min_samples=min_samples,
                algorithm="ball_tree", metric="euclidean").fit(coords)
labels = retain_minimum_area_clusters(
    db.labels_, pixel_area_acres=pixel_area_acres, minimum_acres=DBSCAN_MIN_ACRES
)

n_patches = len(set(labels)) - (1 if -1 in labels else 0)
n_noise   = (labels == -1).sum()
print(f"  Viable patches identified : {n_patches}")
print(f"  Noise pixels (too small)  : {n_noise:,}")

---
## 3. Rank Patches

In [ ]:
# Build summary for each patch
patch_records = []

with rasterio.open(OUTPUTS_DIR / "bhsi_water.tif") as wf:
    water_arr = wf.read(1)
with rasterio.open(OUTPUTS_DIR / "bhsi_soils.tif") as sf:
    soils_arr = sf.read(1)
with rasterio.open(OUTPUTS_DIR / "bhsi_vegetation.tif") as vf:
    veg_arr = vf.read(1)

for patch_id in sorted(set(labels)):
    if patch_id == -1:
        continue
    mask  = labels == patch_id
    p_rows, p_cols = rows[mask], cols[mask]
    p_bhsi  = bhsi_array[p_rows, p_cols]
    p_water = water_arr[p_rows, p_cols] if water_arr is not None else np.array([np.nan])
    p_soils = soils_arr[p_rows, p_cols]
    p_veg   = veg_arr[p_rows, p_cols]

    area_acres = mask.sum() * pixel_area_acres

    # Patch centroid in geographic coords
    cx_m = float(xs[mask].mean())
    cy_m = float(ys[mask].mean())

    patch_records.append({
        "patch_id":        patch_id,
        "n_pixels":        int(mask.sum()),
        "area_acres":      round(area_acres, 0),
        "mean_bhsi":       round(float(p_bhsi.mean()), 3),
        "min_bhsi":        round(float(p_bhsi.min()), 3),
        "mean_water":      round(float(np.nanmean(p_water)), 3),
        "mean_soils":      round(float(np.nanmean(p_soils)), 3),
        "mean_veg":        round(float(np.nanmean(p_veg)), 3),
        "centroid_x_m":    round(cx_m, 0),
        "centroid_y_m":    round(cy_m, 0),
    })

patches_df = pd.DataFrame(patch_records).sort_values("mean_bhsi", ascending=False).reset_index(drop=True)
patches_df["rank"] = patches_df.index + 1

print(f"TOP 15 PRIORITY RESTORATION UNITS")
print("=" * 75)
print(patches_df.head(15)[
    ["rank","patch_id","area_acres","mean_bhsi","mean_water","mean_soils","mean_veg"]
].to_string(index=False))

---
## 4. Convert Patches to Vector Polygons

In [ ]:
# Convert DBSCAN label raster back to polygons for the GeoPackage output
label_raster = np.full(bhsi_array.shape, -1, dtype=np.int32)
for patch_id in sorted(set(labels)):
    if patch_id == -1:
        continue
    mask = labels == patch_id
    label_raster[rows[mask], cols[mask]] = patch_id

# Vectorize patches
with rasterio.open(TEMPLATE_PATH) as tmpl:
    poly_transform = tmpl.transform
    poly_crs       = tmpl.crs

poly_records = []
for geom, value in shapes(label_raster, transform=poly_transform):
    if int(value) < 0:
        continue
    poly_records.append({
        "patch_id":  int(value),
        "geometry":  shape(geom),
    })

if poly_records:
    patches_gdf = gpd.GeoDataFrame(poly_records, crs=poly_crs)
    patches_gdf = patches_gdf.dissolve(by="patch_id", as_index=False)
    patches_gdf = patches_gdf.merge(
        patches_df[["patch_id","rank","area_acres","mean_bhsi",
                    "mean_water","mean_soils","mean_veg"]],
        on="patch_id", how="left"
    )

    # Save GeoPackage
    gpkg_path = OUTPUTS_DIR / "priority_restoration_units.gpkg"
    patches_gdf.to_file(gpkg_path, driver="GPKG", layer="priority_units")

    # Save CSV summary
    csv_path = OUTPUTS_DIR / "priority_restoration_units.csv"
    patches_df.to_csv(csv_path, index=False)
    manifest_path = write_bhsi_manifest(
        OUTPUTS_DIR / "bhsi_provenance.json", available,
        {"threshold_percentile": BHSI_THRESHOLD_PCT, "dbscan_eps_m": DBSCAN_EPS_M,
         "minimum_patch_acres": DBSCAN_MIN_ACRES}
    )

    print(f"Priority units saved:")
    print(f"  GeoPackage : outputs/priority_restoration_units.gpkg")
    print(f"  CSV        : outputs/priority_restoration_units.csv")
    print(f"  Provenance : {manifest_path.name}")
    print(f"  Total units: {len(patches_gdf)}")
    print(f"  Total area in priority units: {patches_df['area_acres'].sum():,.0f} acres")
else:
    print("No polygon patches to vectorize.")

---
## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# BHSI composite map
ax = axes[0]
im = ax.imshow(bhsi_array, cmap="RdYlGn", vmin=0, vmax=1, origin="upper")
plt.colorbar(im, ax=ax, label="BHSI (0=unsuitable, 1=excellent)", shrink=0.8)
ax.set_title(
    "Bison Habitat Suitability Index (BHSI)\n"
    "Composite of 5 components",
    fontsize=11, fontweight="bold",
)
ax.set_xlabel("Column (west -> east)")
ax.set_ylabel("Row (north -> south)")

# Priority patches map
ax = axes[1]
ax.imshow(bhsi_array, cmap="Greys", vmin=0, vmax=1, origin="upper", alpha=0.4)

# Color patches by rank tier
if poly_records and len(patches_gdf) > 0:
    import matplotlib.cm as cm
    cmap_patches = cm.get_cmap("RdYlGn")
    for _, patch in patches_gdf.iterrows():
        color = cmap_patches(patch["mean_bhsi"])
        try:
            from rasterio.features import rasterize as _rast
            pmask = label_raster == patch["patch_id"]
            overlay = np.zeros((*bhsi_array.shape, 4), dtype=np.float32)
            overlay[pmask] = [*color[:3], 0.7]
            ax.imshow(overlay, origin="upper")
        except Exception:
            pass

ax.set_title(
    f"Priority Restoration Units (n={n_patches})\n"
    f"DBSCAN clusters: eps={DBSCAN_EPS_M/1000:.0f}km, min={DBSCAN_MIN_ACRES:.0f}ac",
    fontsize=11, fontweight="bold",
)
ax.set_xlabel("Column (west -> east)")
ax.set_ylabel("Row (north -> south)")

plt.suptitle(
    "Bison Habitat Suitability Index — Pine Ridge, Oglala Lakota Nation\n"
    "Daear Consulting / OLC Cubedynamics",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "07_bhsi_composite.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Component Contribution Summary

In [ ]:
print("BHSI COMPONENT WEIGHTS AND MEAN SCORES")
print("=" * 60)
print(f"  {'Component':<15} {'Weight':>8}  {'Mean Score':>12}  {'Contribution':>14}")
print("  " + "-" * 54)

total_contribution = 0
for comp, weight in BHSI_WEIGHTS.items():
    path = LAYER_PATHS.get(comp)
    if path and path.exists():
        with rasterio.open(path) as src:
            arr = src.read(1)
        mean_score = float(np.nanmean(arr))
    else:
        mean_score = float("nan")
    contribution = weight * mean_score if not np.isnan(mean_score) else 0
    total_contribution += contribution
    print(f"  {comp:<15} {weight*100:>7.0f}%  {mean_score:>12.3f}  {contribution:>14.3f}")

print("  " + "-" * 54)
print(f"  {'BHSI Total':<15} {'100%':>8}  {'':>12}  {total_contribution:>14.3f}")
print()
print(f"Mean BHSI across Pine Ridge: {np.nanmean(bhsi_array):.3f}")
print()
print("INTERPRETATION:")
if np.nanmean(bhsi_array) >= 0.6:
    print("  HIGH: Substantial portion of Pine Ridge is suitable or restorable.")
elif np.nanmean(bhsi_array) >= 0.4:
    print("  MODERATE: Significant habitat exists with restoration investment needed.")
else:
    print("  LOW: Most land requires restoration — identify priority patches first.")

In [ ]:
# Sensitivity: show how top-ranked patches change under different weights
scenarios = {
    "Default":          BHSI_WEIGHTS,
    "Water-critical":   {"vegetation":0.25,"soils":0.20,"water":0.35,"topography":0.10,"climate":0.10},
    "Vegetation-heavy": {"vegetation":0.40,"soils":0.20,"water":0.20,"topography":0.15,"climate":0.05},
    "Equal weights":    {"vegetation":0.20,"soils":0.20,"water":0.20,"topography":0.20,"climate":0.20},
}

print("SENSITIVITY ANALYSIS — Top 5 patches under different weight scenarios")
print("=" * 70)

for scenario_name, weights in scenarios.items():
    alt_bhsi = combine_bhsi_layers(available, weights=weights)
    alt_threshold = np.nanpercentile(alt_bhsi, BHSI_THRESHOLD_PCT)
    alt_high = (alt_bhsi > alt_threshold) & (~np.isnan(alt_bhsi))
    alt_rows, alt_cols = np.where(alt_high)
    if len(alt_rows) == 0:
        print(f"  {scenario_name}: no high-suitability pixels found")
        continue
    alt_xs = transform.c + alt_cols * transform.a
    alt_ys = transform.f + alt_rows * transform.e
    alt_coords = np.column_stack([alt_xs, alt_ys])
    alt_db = DBSCAN(eps=DBSCAN_EPS_M, min_samples=min_samples,
                    algorithm="ball_tree").fit(alt_coords)
    alt_labels = retain_minimum_area_clusters(
        alt_db.labels_, pixel_area_acres=pixel_area_acres, minimum_acres=DBSCAN_MIN_ACRES
    )
    alt_n = len(set(alt_labels)) - (1 if -1 in alt_labels else 0)
    overlap = (alt_high & high_suit).sum() / (alt_high | high_suit).sum()
    print(f"  {scenario_name}: {alt_n} patches | high-suitability spatial overlap with default: {overlap:.1%}")

print()
print("Higher spatial overlap indicates a more robust high-suitability footprint.")

In [ ]:
# Final export confirmation
print("FINAL OUTPUTS")
print("=" * 55)
outputs = [
    ("outputs/bhsi_composite.tif", "BHSI raster — pixel-level scores"),
    ("outputs/priority_restoration_units.gpkg", "Priority units — GeoPackage for GIS"),
    ("outputs/priority_restoration_units.csv",  "Priority units — ranked summary table"),
    ("outputs/figures/07_bhsi_composite.png",   "BHSI composite map"),
]
for fname, desc in outputs:
    path = REPO_ROOT / fname
    exists = "✓" if path.exists() else "✗ (not found)"
    print(f"  {exists}  {fname}")
    print(f"       {desc}")

print()
print("NEXT STEPS FOR OLC CUBEDYNAMICS:")
print("  1. Review priority_restoration_units.gpkg in GIS")
print("  2. Ground-truth top 5 patches with field observation")
print("  3. Overlay with land ownership and existing infrastructure")
print("  4. Confirm DBSCAN_MIN_ACRES threshold with bison program")
print("  5. Adjust BHSI weights in src/constants.py based on local knowledge")
print()
print("All results should be reviewed by OLC and the Oglala Lakota Nation")
print("land management offices before external distribution.")
print()
print(generate_citations(["census_aiannh","modis_ndvi","nlcd","gssurgo","dem_3dep","nhd","maca_climate"]))